# A2 2025 - Template de Estudo
### Projeto e Análise de Algoritmos — FGV EMAp

Template para estudar o conteúdo da segunda avaliação (grafos, paradigmas de projeto e classes de complexidade), no mesmo formato do notebook da A1: uma célula de **enunciado** por questão, uma célula de **código** com a assinatura da função e uma célula de **testes** com os exemplos. Respostas discursivas ficam em células markdown marcadas com **Resposta:**.

| Parte | Prova | Questões (pontos) |
|---|---|---|
| 1 | A2 2025.2 — Prova | Q1 (2,5) · Q2 (3,0) · Q3 (3,0) · Q4 (1,5) |
| 2 | A2 2025.2 — Simulado | Q1 (3) · Q2 (2) · Q3 (3) · Q4 (2) |
| 3 | A2 2024.2 — Prova | Q3 (3) — as Q1, Q2 e Q4 são as mesmas do Simulado |

> **Convenções (regras da prova):** não altere o nome do arquivo nem as assinaturas das funções; classes ou funções auxiliares devem ser definidas **dentro** do escopo da função da questão; algoritmos devem ser codificados e as demais respostas escritas nos comentários/células indicadas.
>
> **Sobre as assinaturas:** os arquivos `.py` originais (`paa2025_A2_prova.py`, `paa2025_A2_simulado.py`) não estavam disponíveis, então os nomes e parâmetros abaixo foram inferidos dos enunciados e dos exemplos. Ajuste-os se tiver os arquivos originais. Os exemplos marcados como *autorais* não vêm das provas.

---
## Parte 1 — A2 2025.2 · Prova

A empresa Emezon está criando um novo sistema de entregas autônomo em uma cidade, e para tanto é necessário desenvolver um conjunto de serviços para apoiar a operação. Você foi nomeado para projetar e implementar os algoritmos a serem utilizados por esses serviços.

A prefeitura de cada cidade irá fornecer uma versão digital da planta da cidade. Essa planta representa ruas através de uma sequência de segmentos. Cada segmento possui em sua extremidade um cruzamento, que realiza a conexão com outros segmentos da mesma ou de outras ruas. Toda cidade que receberá o serviço possui de centenas a milhares de ruas, todas de mão dupla, e nenhum cruzamento conecta mais de 4 segmentos.

---
### Questão 01 (2,5 pontos)
A infraestrutura de ruas da cidade terá que ser ligeiramente modificada para acomodar uma frota de veículos autônomos. Para tanto, será necessário escolher quais segmentos receberão tal adaptação. O custo de adaptar um segmento varia de acordo com características da rua (dimensões, tipo de asfalto, inclinação, etc). Felizmente um engenheiro de cada cidade forneceu o custo de adaptar cada segmento. Portanto, dada uma lista de cruzamentos associando cada um aos cruzamentos adjacentes, produza um algoritmo capaz de verificar se, dado um orçamento $C$, é possível selecionar um conjunto de segmentos para adaptação de forma que a estrutura resultante possibilite rotas entre qualquer par de cruzamentos da cidade utilizando apenas esses segmentos adaptados. Analise a complexidade do algoritmo produzido.

**Exemplos:** os segmentos são representados por `(cruzamento_1, cruzamento_2, custo)`:

```python
[(1, 2, 1), (1, 4, 2), (2, 3, 7),
 (2, 5, 1), (3, 6, 9), (4, 5, 3),
 (4, 7, 2), (5, 6, 1), (5, 8, 2),
 (6, 9, 10), (7, 8, 1), (8, 9, 1)]
```

| Entrada | Saída |
|---|---|
| `C = 16` | `True` |
| `C = 21` | `True` |
| `C = 10` | `False` |

In [9]:
def questao_01(segmentos: list, C: int) -> bool:
	"""Verifica se, com orçamento C, é possível adaptar um conjunto de segmentos que conecte todos os cruzamentos.

	Parameters
	----------
	segmentos : list
		Lista de tuplas (cruzamento_1, cruzamento_2, custo). Ruas de mão dupla.
	C : int
		Orçamento disponível.

	Returns
	-------
	bool
		True se existe um conjunto de segmentos com custo total <= C que permite rota entre qualquer par de cruzamentos.
	"""
	import heapq

	def get_cost(root, adjacencias, n):
		visitados = set([root])
		heap = adjacencias[root]
		heapq.heapify(heap)
		cost = 0

		while len(visitados) < n and len(heap) > 0:
			cost_v, v = heapq.heappop(heap)
			if v in visitados:
				continue

			visitados.add(v)
			cost += cost_v
			for c, u in adjacencias[v]:
				heapq.heappush(heap, (c,u))

		return cost

	INF = float('inf')
	min_cost = INF
	min_node = -1
	adjacencias = {}
	for u, v, c in segmentos:
		if c < min_cost:
			min_node = u

		if u in adjacencias:
			adjacencias[u].append((c,v))
		else:
			adjacencias[u] = [(c,v)]

		if v in adjacencias:
			adjacencias[v].append((c,u))
		else:
			adjacencias[v] = [(c,u)]

	return True if get_cost(min_node, adjacencias, len(adjacencias)) <= C else False


In [10]:
segmentos = [(1, 2, 1), (1, 4, 2), (2, 3, 7),
			 (2, 5, 1), (3, 6, 9), (4, 5, 3),
			 (4, 7, 2), (5, 6, 1), (5, 8, 2),
			 (6, 9, 10), (7, 8, 1), (8, 9, 1)]

print(questao_01(segmentos, 16))  # esperado: True
print(questao_01(segmentos, 21))  # esperado: True
print(questao_01(segmentos, 10))  # esperado: False

True
True
False


**Resposta:**

**Análise de complexidade:** _(escreva aqui)_

---
### Questão 02 (3,0 pontos)
O computador de bordo do veículo precisa encontrar o caminho entre dois cruzamentos. A infraestrutura das ruas modificadas é capaz de alimentar a bateria do veículo em alguns segmentos. Considerando que sabemos o consumo de energia em cada segmento, crie um algoritmo capaz de descobrir o caminho entre dois cruzamentos de forma que o saldo da bateria ($consumo - recarga$) seja mínimo (ou seja, chegue ao destino com o maior valor possível). Considere que o sistema impede que o veículo percorra ciclos de recarga indefinidamente, ou seja, impede que os segmentos adaptados formem ciclos cujo saldo total seja negativo.

Observe que o consumo negativo representa recarga da bateria.

**Exemplo:** os segmentos são representados por `(cruzamento_1, cruzamento_2, consumo)`:

```python
[(1, 2, 2), (1, 4, -2), (2, 3, 2),
 (2, 5, -3), (4, 7, 2), (5, 6, 3),
 (5, 8, 5), (7, 8, -2), (8, 9, 1)]
```

Ao buscar o caminho com maior eficiência energética entre o cruzamento 3 e o cruzamento 9 temos como solução `[(3, 2), (2, 1), (1, 4), (4, 7), (7, 8), (8, 9)]`.

> **Nota do template:** o enunciado não detalha como o consumo negativo se aplica ao sentido de percurso de cada segmento (no exemplo, o trecho `(2, 3)` é percorrido como `(3, 2)`). Defina e documente sua interpretação antes de implementar.

In [ ]:
def questao_02(segmentos: list, origem: int, destino: int) -> list:
	"""Encontra o caminho de menor saldo de bateria (consumo - recarga) entre dois cruzamentos.

	Parameters
	----------
	segmentos : list
		Lista de tuplas (cruzamento_1, cruzamento_2, consumo); consumo negativo representa recarga.
	origem : int
		Cruzamento de partida.
	destino : int
		Cruzamento de chegada.

	Returns
	-------
	list
		Sequência de tuplas (u, v) percorridas da origem ao destino.
		Defina o retorno para o caso sem caminho (ex.: None) e documente.
	"""

	
	

In [ ]:
segmentos = [(1, 2, 2), (1, 4, -2), (2, 3, 2),
			 (2, 5, -3), (4, 7, 2), (5, 6, 3),
			 (5, 8, 5), (7, 8, -2), (8, 9, 1)]

print(questao_02(segmentos, 3, 9))  # esperado: [(3, 2), (2, 1), (1, 4), (4, 7), (7, 8), (8, 9)]

---
### Questão 03 (3,0 pontos)
O veículo autônomo terá um mecanismo para que a taxa de entrega seja paga pelo destinatário, no entanto é necessário calcular o troco automaticamente com base nos tipos de moedas disponíveis. Crie um algoritmo que receba o valor a ser pago de troco e a lista com os valores disponíveis de moedas, e retorne quais moedas deverão ser utilizadas, buscando a menor quantidade possível. Analise a complexidade do algoritmo produzido.

Observe que esse problema é semelhante a um problema apresentado em sala de aula.

**Exemplos:**

| Entrada | Saída |
|---|---|
| `coins = [1, 5, 6, 9]` e `value = 11` | `[5, 6]` |
| `coins = [2]` e `value = 3` | `-1` |
| `coins = [1, 2, 5]` e `value = 11` | `[1, 5, 5]` |

In [27]:
def questao_03(coins: list, value: int):
	"""Retorna as moedas usadas para formar `value` com a menor quantidade possível de moedas.

	Parameters
	----------
	coins : list
		Valores de moedas disponíveis (cada valor pode ser usado quantas vezes for necessário).
	value : int
		Valor do troco.

	Returns
	-------
	list | int
		Lista com as moedas utilizadas, ou -1 se não for possível formar o valor.
	"""
	n = len(coins)
	memo = {coins[0]: [v // coins[0] if v % coins[0] == 0 else -1 for v in range(value+1)]}
	usage = {coins[0]: [[(v // coins[0], coins[0])] if v % coins[0] == 0 else [(0, coins[0])] for v in range(value+1)]}
	for i in range(1, n):
		atual = coins[i]
		prev = coins[i-1]
		memo[atual] = memo[prev].copy()
		usage[atual] = usage[prev].copy()
		# temp data ^

		for val in range(value+1):
			max_usage = val // atual
			for j in range(max_usage+1):
				if j + memo[prev][val-j*atual] < memo[prev][val]:
					memo[atual][val] = j + memo[prev][val-j*atual]
					usage[atual][val] = usage[prev][val-j*atual] + [(j, atual)]

	final = list()
	for u, c in usage[coins[-1]][value]:
		final += u*[c]

	return final if len(final) > 0 else -1

In [28]:
print(questao_03([1, 5, 6, 9], 11))  # esperado: [5, 6]
print(questao_03([2], 3))            # esperado: -1
print(questao_03([1, 2, 5], 11))     # esperado: [1, 5, 5]

[5, 6]
-1
[1, 5, 5]


**Resposta:**

**Análise de complexidade:** _(escreva aqui)_

---
### Questão 04 (1,5 pontos)
Um gestor da operação pediu para criar um algoritmo <u>eficiente</u> capaz de calcular a rota de menor custo passando por todos os pontos de entrega de forma que nunca precise passar por ele novamente.

**a)** Qual a sua opinião sobre a viabilidade desse pedido?

**b)** Qual seria a explicação teórica para embasar essa opinião?

**c)** O que seria necessário para construir tal solução?

Apresente uma explicação com o maior grau de profundidade que você conseguir.

**Resposta:**

**a)** _(escreva aqui)_

**b)** _(escreva aqui)_

**c)** _(escreva aqui)_

---
## Parte 2 — A2 2025.2 · Simulado

---
### Questão 01 (3 pontos)
Uma organização ambiental desenvolveu um sistema para monitoramento de florestas cuja comunicação é baseada em uma rede de sensores sem fio. Um sensor $A$ consegue enviar uma mensagem para um sensor $B$ diretamente se a distância entre eles for menor ou igual ao raio de transmissão de $A$. Dado um conjunto de sensores $S$, em que cada elemento $s$ possui uma localização $(s.x, s.y)$ e um raio de transmissão $s.r$ em metros, projete um algoritmo capaz de calcular a rota para transmitir uma mensagem entre $s_i$ e $s_f$ através dos sensores da rede, de forma que percorra a menor distância possível em metros. O algoritmo deve retornar uma sequência de sensores representando o caminho completo que a mensagem irá percorrer: primeiro os sensores utilizados para enviar a requisição de $s_i$ até $s_f$, e em seguida os sensores utilizados para enviar a resposta de $s_f$ até $s_i$. Caso não seja possível estabelecer a comunicação, o algoritmo deve indicar isso. Analise a complexidade do algoritmo proposto.

> O enunciado não traz exemplos. Os casos de teste abaixo são *autorais*.

In [15]:
def questao_01(sensores: list, i: int, f: int):
	"""Calcula a rota de menor distância para requisição (i -> f) seguida da resposta (f -> i).

	Parameters
	----------
	sensores : list
		Lista de tuplas (x, y, r): posição e raio de transmissão (em metros) de cada sensor;
		o sensor k é sensores[k].
	i : int
		Índice do sensor de origem (s_i).
	f : int
		Índice do sensor de destino (s_f).

	Returns
	-------
	list | None
		Sequência de índices de sensores: primeiro a ida (i -> f), depois a volta (f -> i).
		None se a comunicação (ida ou volta) não for possível. Documente como representa o sensor
		de junção (f) na sequência.
	"""
	import heapq

	def dist(s1, s2):
		return ((s1[0]-s2[0])**2+(s1[1]-s2[1])**2)**(1/2)

	def dijkstra(i, f, n, adjacencias):
		if not i in adjacencias:
			return None

		INF = float('inf')
		parents = [-1 for _ in range(n)]
		costs = [INF for _ in range(n)]
		costs[i] = 0

		visitados = set([i])
		heap = [(1, v, i) for v in adjacencias[i]]
		heapq.heapify(heap)
		while len(visitados) < n and len(heap) > 0:
			c, v, pai = heapq.heappop(heap)
			if v in visitados:
				continue

			visitados.add(v)
			parents[v] = pai
			costs[v] = c

			if v not in adjacencias:
				continue

			for u in adjacencias[v]:
				heapq.heappush(heap, (c+1, u, v))

		if costs[f] == INF:
			return None

		path = [f]
		pai = parents[f]
		while pai != -1:
			path.append(pai)
			pai = parents[pai]

		path.reverse()
		return path

	n = len(sensores)
	adjacencias = {}
	for k in range(n):
		for j in range(k+1, n):
			d = dist(sensores[k], sensores[j])

			if d < sensores[k][2]:
				if k in adjacencias:
					adjacencias[k].append(j)
				else:
					adjacencias[k] = [j]

			if d < sensores[j][2]:
				if j in adjacencias:
					adjacencias[j].append(k)
				else:
					adjacencias[j] = [k]

	ida = dijkstra(i, f, n, adjacencias)
	if ida is None: return None

	volta = dijkstra(f, i, n, adjacencias)
	if volta is None: return None

	return ida + volta

In [16]:
# Casos autorais
# 1) Sensores em linha, raio 5: 0 alcança 1, 1 alcança 0 e 2, 2 alcança 1 (0 e 2 não se alcançam direto)
s = [(0, 0, 5), (3, 0, 5), (6, 0, 5)]
print(questao_01(s, 0, 2))  # esperado: ida [0, 1, 2] e volta [2, 1, 0]

# 2) Alcance assimétrico: 0 alcança 1 (8 <= 10), mas 1 não alcança 0 (8 > 2) -> a resposta não volta
s = [(0, 0, 10), (8, 0, 2)]
print(questao_01(s, 0, 1))  # esperado: None

# 3) Sensores fora de alcance um do outro
s = [(0, 0, 1), (10, 0, 1)]
print(questao_01(s, 0, 1))  # esperado: None

[0, 1, 2, 2, 1, 0]
None
None


**Resposta:**

**Análise de complexidade:** _(escreva aqui)_

---
### Questão 02 (2 pontos)
Considere um grafo $G = (V, E)$ conexo e não-dirigido. Dizemos que uma aresta $e \in E$ é uma ponte se a sua remoção produzir um grafo $G'$ não-conexo.

**a)** Se existir uma aresta $e = (v_i, v_j)$ que não é uma ponte podemos afirmar que existe um ciclo em $G$ que contém os vértices $v_i$ e $v_j$. Por que?

**b)** Projete um algoritmo que receba uma aresta $e = (v_i, v_j)$ e determina se a mesma é uma ponte em $O(V + E)$.

> Na prova de 2024.2 (mesma questão, sem o limite $O(V+E)$ em (b)), os algoritmos de aula usados na solução deviam ser **descritos explicitamente**, não apenas referenciados.

**Resposta:**

**a)** _(escreva aqui)_

In [20]:
def questao_02_b(n: int, arestas: list, e: tuple) -> bool:
	"""Determina se a aresta e é uma ponte do grafo conexo e não-dirigido G, em O(V + E).

	Parameters
	----------
	n : int
		Número de vértices (rotulados de 0 a n-1).
	arestas : list
		Lista de tuplas (u, v) com as arestas de G.
	e : tuple
		Aresta (v_i, v_j) a ser testada (pertence a `arestas`).

	Returns
	-------
	bool
		True se e é uma ponte (sua remoção desconecta G), False caso contrário.
	"""
	adjacencias = {v: list() for v in range(n)}
	for u,v in arestas:
		adjacencias[u].append(v)
		adjacencias[v].append(u)

	fila = [e[0]]
	visitados = set()
	atual = 0
	while len(fila) > atual:
		v = fila[atual]
		atual+=1
		if v in visitados:
			continue

		visitados.add(v)
		for u in adjacencias[v]:
			if (u,v) != e and (v,u) != e:
				fila.append(u)

	return len(visitados) != n


In [21]:
# Casos autorais: triângulo 0-1-2 com um pendente 2-3
arestas = [(0, 1), (1, 2), (0, 2), (2, 3)]
print(questao_02_b(4, arestas, (2, 3)))  # esperado: True
print(questao_02_b(4, arestas, (0, 1)))  # esperado: False
print(questao_02_b(4, arestas, (1, 2)))  # esperado: False

True
False
False


**Resposta:**

**Análise de complexidade de (b):** _(escreva aqui)_

---
### Questão 03 (3 pontos)
Considere o problema de agendamento de tarefas: dado o conjunto de tarefas $T = \{t_1, .., t_n\}$ com $n$ elementos, cada uma com um tempo de início $t_k.start$, um tempo de término $t_k.end$, e um valor $t_k.value$, encontre o subconjunto de tarefas $S \subseteq T$ que pode ser alocado sem sobreposição temporal de forma que o valor $\sum_{i=1}^{|S|} t_i.value$ seja máximo.

**a)** Projete um algoritmo (auxiliar) que receba como parâmetro uma tarefa e retorne a tarefa anterior compatível (ou seja, que pode ser alocada sem sobreposição temporal) com o tempo de término mais próximo.

**b)** Projete um algoritmo para agendamento de tarefas que produza sempre a solução ótima, utilizando o algoritmo auxiliar criado em (a). O algoritmo deverá respeitar as características descritas no enunciado acima.

**c)** Qual técnica de projeto você utilizou para projetar o algoritmo de (b)?

**d)** Analise a complexidade do algoritmo e justifique a sua resposta.

OBS: Esse problema é semelhante a um problema apresentado em aula. Avalie atentamente as diferenças para encontrar a solução adequada.

> O enunciado não traz exemplos. Os casos de teste abaixo são *autorais* (tarefas como `(start, end, value)`; sem tarefas que se tocam nas extremidades, para evitar ambiguidade).

In [ ]:
def questao_03_a(tarefas: list, k: int) -> int:
	"""Retorna a tarefa anterior compatível com a tarefa k cujo tempo de término é o mais próximo do início de k.

	Parameters
	----------
	tarefas : list
		Lista de tuplas (start, end, value). Defina e documente a pré-condição de ordenação.
	k : int
		Índice da tarefa de referência.

	Returns
	-------
	int
		Índice da tarefa anterior compatível, ou -1 se não existir.
	"""
	pass


def questao_03_b(tarefas: list) -> tuple:
	"""Encontra o subconjunto de tarefas sem sobreposição com valor total máximo.

	Parameters
	----------
	tarefas : list
		Lista de tuplas (start, end, value).

	Returns
	-------
	tuple
		(valor_total_maximo, tarefas_escolhidas), com tarefas_escolhidas como lista de índices em `tarefas`.
	"""
	pass

In [ ]:
# Casos autorais
t1 = [(1, 3, 5), (2, 5, 6), (4, 7, 5), (6, 9, 4), (8, 10, 3)]
print(questao_03_b(t1))  # esperado: valor 13 (tarefas 0, 2 e 4)

# Uma tarefa longa e valiosa vence várias curtas (guloso por término daria 30)
t2 = [(1, 10, 100), (2, 3, 10), (4, 5, 10), (6, 7, 10)]
print(questao_03_b(t2))  # esperado: valor 100 (tarefa 0)

**Resposta:**

**c)** _(escreva aqui)_

**d)** _(escreva aqui)_

---
### Questão 04 (2 pontos)
Para cada uma das afirmações abaixo indique se é verdadeira (V) ou falsa (F):

( ) Se um problema A pode ser reduzido a um problema B em tempo polinomial, e B está em P, então A também está em P.

( ) Todo problema NP-Difícil é necessariamente NP-Completo.

( ) Se encontrarmos um algoritmo polinomial para um problema NP-Completo, isso prova que P = NP.

( ) Se tivermos um problema A que é NP-Completo e um problema B que é NP-Difícil, e A pode ser reduzido a B em tempo polinomial, então B é necessariamente NP-Completo.

( ) Um problema pode ser simultaneamente NP-Completo e estar em P apenas se P = NP.

**Resposta:**

| # | V/F | Justificativa |
|---|---|---|
| 1 | ( ) | _(escreva aqui)_ |
| 2 | ( ) | _(escreva aqui)_ |
| 3 | ( ) | _(escreva aqui)_ |
| 4 | ( ) | _(escreva aqui)_ |
| 5 | ( ) | _(escreva aqui)_ |

---
## Parte 3 — A2 2024.2 · Prova

As Questões 1, 2 e 4 desta prova são as mesmas do Simulado 2025.2 (Parte 2); a Questão 3 é diferente e está abaixo.

---
### Questão 03 (3 pontos)
Uma empresa está projetando a infraestrutura de comunicação para sua nova planta industrial. A planta possui diversos prédios que precisam ser conectados através de uma rede de fibra óptica. Para cada par de prédios, existe um custo associado à instalação da fibra e uma largura de banda máxima que essa conexão pode atingir (determinada pela distância e características do terreno entre os prédios). A equipe de engenharia já definiu o conjunto mínimo de conexões necessárias para interligar todos os prédios minimizando o custo total, no entanto a equipe de TI gostaria de tornar o projeto tolerante à falhas introduzindo redundâncias - ou seja, para cada par de prédios deveria existir mais de um caminho na rede. Dada a topologia completa da planta (com os custos e larguras de banda possíveis para cada conexão), o conjunto de conexões já escolhido, e um requisito mínimo de velocidade $W$ para a rede, projete um algoritmo que decide se é possível tornar a rede tolerante à falhas, garantindo que sempre existirá uma rota entre dois prédios caso uma conexão falhe. O algoritmo deverá retornar a lista de conexões redundantes respeitando o requisito de velocidade $W$, ou indicar que não é possível garantir a redundância para todas as conexões. Analise a complexidade do algoritmo proposto.

Caso seja necessário, é permitido nessa questão fazer referência a estruturas de dados e algoritmos apresentados em aula para compor a solução.

> O enunciado não traz exemplos nem assinatura; os parâmetros abaixo foram inferidos.

In [ ]:
def questao_03(n: int, conexoes: list, escolhidas: list, W: int):
	"""Decide se é possível tornar a rede tolerante a falhas e retorna as conexões redundantes.

	Parameters
	----------
	n : int
		Número de prédios (rotulados de 0 a n-1).
	conexoes : list
		Topologia completa: tuplas (u, v, custo, banda) com todas as conexões possíveis.
	escolhidas : list
		Conexões já escolhidas pela engenharia (tuplas (u, v)), que ligam todos os prédios com custo mínimo.
	W : int
		Largura de banda mínima exigida para as conexões adicionadas.

	Returns
	-------
	list | None
		Lista de conexões redundantes (respeitando W) que garantem rota entre quaisquer dois prédios
		mesmo se uma conexão falhar; None se não for possível. Documente a representação escolhida.
	"""
	pass

**Resposta:**

**Análise de complexidade:** _(escreva aqui)_